In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from math import gamma

plt.rcParams.update({
    "text.usetex":        False,
    "font.family":        "serif",
    "font.serif":         ["Palatino", "Palatino Linotype", "Georgia",
                           "Times New Roman", "DejaVu Serif"],
    "mathtext.fontset":   "cm",
    "axes.labelsize":     11,
    "axes.titlesize":     11,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
})

# ── Thesis colour palette ─────────────────────────────────────────────────────
TROPICAL     = (0.0, 0.46, 0.37)              # tropicalrainforest
CTRED        = (177/255, 56/255, 69/255)       # CTRed
CLASSICYELLOW = (191/255, 156/255, 74/255)     # ClassicYellow
LABEL_GREY   = (0.6, 0.6, 0.6)

def _pale(rgb, mix=0.60):
    return tuple(c + (1 - c) * mix for c in rgb)

def _dark(rgb, fac=0.62):
    return tuple(c * fac for c in rgb)

def asymptotic_mean(n_arr, p, q):
    alpha = 2*p - 1
    beta  = 2*q - 1
    return beta * n_arr.astype(float)**alpha / gamma(alpha + 1)

def simulate_erw(n, p, q=0.5, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    xi = np.empty(n + 1, dtype=np.int8)
    xi[0] = 0
    xi[1] = 1 if rng.random() < q else -1
    for t in range(1, n):
        past = xi[rng.integers(0, t) + 1]
        xi[t + 1] = past if rng.random() < p else -past
    S = np.empty(n + 1, dtype=np.int64)
    S[0] = 0
    S[1:] = np.cumsum(xi[1:])
    return S

def gradient_path(ax, x, y, cmap, vmax, lw=0.35, alpha=0.78):
    """Plot a path coloured by distance from zero — pale near origin, vivid far."""
    pts  = np.array([x, y], dtype=float).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    vals = (np.abs(y[:-1]) + np.abs(y[1:])) / 2.0
    norm = mcolors.Normalize(vmin=0, vmax=vmax)
    lc   = LineCollection(segs, cmap=cmap, norm=norm,
                          linewidth=lw, alpha=alpha, rasterized=True)
    lc.set_array(vals)
    ax.add_collection(lc)

# ── Parameters ────────────────────────────────────────────────────────────────
P       = 0.65
N_STEPS = 1000
N_THIN  = 8
N_MEAN  = 500

PANELS = [
    (0.20, TROPICAL,      r"$q = 0.20\ \ (\beta < 0)$"),
    (0.50, CLASSICYELLOW, r"$q = 0.50\ \ (\beta = 0)$"),
    (0.80, CTRED,         r"$q = 0.80\ \ (\beta > 0)$"),
]

steps_plot = np.arange(0, N_STEPS + 1)
steps_th   = np.arange(1, N_STEPS + 1)
rng = np.random.default_rng(314)

# ── Simulate ──────────────────────────────────────────────────────────────────
data = {}
for q_val, base_col, _ in PANELS:
    thin = [simulate_erw(N_STEPS, p=P, q=q_val, rng=rng) for _ in range(N_THIN)]
    mat  = np.array([simulate_erw(N_STEPS, p=P, q=q_val, rng=rng)
                     for _ in range(N_MEAN)], dtype=float)
    data[q_val] = {"thin": thin, "emp": mat.mean(axis=0)}

# ── Shared y-limits ───────────────────────────────────────────────────────────
all_vals = []
for q_val, _, _ in PANELS:
    for S in data[q_val]["thin"]:
        all_vals.extend([S.min(), S.max()])
    all_vals.extend([data[q_val]["emp"].min(), data[q_val]["emp"].max()])
global_min = min(all_vals)
global_max = max(all_vals)
margin = max(abs(global_min), abs(global_max)) * 0.08
ylim = (global_min - margin, global_max + margin)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.9), sharey=True)
fig.subplots_adjust(left=0.07, right=0.97, bottom=0.18, top=0.82, wspace=0.18)

for ax, (q_val, base_col, title) in zip(axes, PANELS):
    cmap  = mcolors.LinearSegmentedColormap.from_list(
                f"c{q_val}", [_pale(base_col), base_col])
    dark  = _dark(base_col)
    thin  = data[q_val]["thin"]
    emp   = data[q_val]["emp"]
    asymp = asymptotic_mean(steps_th, P, q_val)
    vmax  = max(abs(ylim[0]), abs(ylim[1]))

    for S in thin:
        gradient_path(ax, steps_plot, S.astype(float), cmap=cmap, vmax=vmax)

    # empirical mean (solid)
    ax.plot(steps_plot, emp,
            color=dark, lw=0.80, alpha=0.80, zorder=3,
            solid_capstyle="round")

    # asymptotic mean (dashed)
    ax.plot(steps_th, asymp,
            color=dark, lw=0.80, alpha=0.80, zorder=4,
            ls=(0, (5, 3)), solid_capstyle="round")

    ax.set_xlim(0, N_STEPS)
    ax.set_ylim(*ylim)

    ax.axhline(0, color="black", lw=0.45, ls=(0, (4, 3)), alpha=0.28, zorder=0)

    ax.set_title(title, fontsize=9, pad=5, color=LABEL_GREY)
    ax.set_xlabel(r"$n$", labelpad=3)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.55)
    ax.spines["bottom"].set_linewidth(0.55)
    ax.tick_params(width=0.55, length=3, direction="out")

    ax.xaxis.set_major_locator(ticker.FixedLocator([0, N_STEPS // 2, N_STEPS]))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(
        lambda x, _, ns=N_STEPS: f"${int(x)}$"))
    ax.yaxis.set_major_locator(ticker.MaxNLocator(nbins=4, symmetric=True))

    if ax is axes[0]:
        ax.set_ylabel(r"$S_n$", labelpad=4)

fig.text(0.52, 0.96, r"$p = 0.65\ \ (\alpha = 0.3 > 0)$",
         ha="center", va="top", fontsize=9, color=LABEL_GREY)


print("Done")